# Analysis of Misclassifications

In [39]:
from pathlib import Path
import json
import pickle

import pandas as pd

## Compute Misclassifications

### Experiment configuration

In [40]:
# DARPA2000

dataset = "darpa2000"
scenario = "s1_inside"

logic_file = "darpa_test_one_net"

train_mode = "scratch" 
# train_mode = "pretrained"

feature_group = "behavioral"
 
subset = "500b20a"

window_size = 10

run_id = "20260418_131616"

experiment_name = f"{logic_file}_{train_mode}_{feature_group}_w{window_size}_{subset}"
cache_file_name = f"{logic_file}_w{window_size}_{subset}_test.pkl"

In [41]:
# # AIT Log v2

# dataset = "aitv2"
# scenario = "fox"
# # scenario = "santos"

# logic_file = "ait_neg"

# train_mode = "scratch" 
# # train_mode = "pretrained" 

# fraction = 25
# window_size = 10
# run_id = "20260409_165607"

# experiment_name = f"{logic_file}_{train_mode}_{fraction}data_w{window_size}"
# cache_file_name = f"{logic_file}_{fraction}data_w{window_size}_test.pkl"

### Load Original Flows

In [42]:
scenario_parts = scenario.split("_")
if len(scenario_parts) == 3:
    test_set_tag = f"{scenario_parts[0]}_{scenario_parts[2]}"
else:
    test_set_tag = scenario

In [43]:
df = pd.read_csv(
    f"../data/interim/{dataset}/{test_set_tag}/flows_labeled/all_flows_labeled.csv"
)

df = df.sort_values("start_time").reset_index(drop=True)
df['t_rel'] = df['start_time'] - df['start_time'].min()

### Load DPL Dataset

In [44]:
def load_dpl_dataset(logic_file, cache_file_name):
    dpl_dataset_dir = Path(f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/cache/")
    cache_file_test = dpl_dataset_dir / cache_file_name

    cache = pickle.load(open(cache_file_test, "rb"))
    cache_df = pd.DataFrame(cache)
    cache_df.head()

    return cache_df

In [45]:
cache_df = load_dpl_dataset(logic_file, cache_file_name)

### Map Misclassifications to Original Flows

In [46]:
errors_dir = f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/errors"
errors_file = (
    f"{errors_dir}/"
    f"{experiment_name}_{run_id}.json"
)
    
with open(errors_file) as f:
    errors = json.load(f)

In [47]:
dpl_to_orig = dict(zip(cache_df['dpl_index'], cache_df['orig_index']))

original_indices = []
mis_y_preds = []
mis_y_trues = []

phase_map = {
    "benign": 0,
    "phase1": 1,
    "phase2": 2,
    "phase3": 3,
    "phase4": 4,
    "phase5": 5,
}

for error in errors:
    dpl_index = error['index']

    original_indices.append(dpl_to_orig[dpl_index])
    y_pred = error['predicted']
    y_true = error['actual']
    mis_y_preds.append(phase_map[y_pred])
    mis_y_trues.append(phase_map[y_true])

mis_df = df.loc[original_indices].copy()
mis_df['y_pred'] = mis_y_preds
mis_df['y_true'] = mis_y_trues

In [48]:
mis_df

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
21650,f21931,9.524407e+08,9.524407e+08,24.675615,172.16.112.194,0,202.77.162.213,8,icmp,-,...,11,418,1,38,-,1,1,1799.665644,0,1
23250,f23446,9.524413e+08,9.524413e+08,0.064471,135.13.216.191,8,172.16.113.169,0,icmp,-,...,9,342,0,0,-,1,0,2405.107653,1,0
24136,f24195,9.524417e+08,9.524417e+08,0.121346,202.77.162.213,54792,172.16.115.20,32773,udp,-,...,1,100,1,52,-,17,2,2790.882799,0,2
25735,f25926,9.524421e+08,9.524421e+08,0.057435,202.77.162.213,56256,172.16.112.10,32774,udp,-,...,1,100,1,52,-,17,2,3213.549749,0,2
29151,f29122,9.524433e+08,9.524433e+08,4.323372,202.77.162.213,46987,172.16.112.10,23,tcp,-,...,14,622,13,762,-,6,3,4392.243433,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114659,f114639,9.524470e+08,9.524470e+08,0.070200,172.16.115.5,23718,131.84.1.31,80,tcp,http,...,6,508,7,3592,-,6,5,8103.175602,0,5
114702,f114643,9.524470e+08,9.524470e+08,0.165900,172.16.116.201,23972,131.84.1.31,80,tcp,http,...,14,773,29,38417,-,6,5,8107.951338,0,5
115033,f115007,9.524471e+08,9.524471e+08,0.078300,172.16.115.5,24713,131.84.1.31,80,tcp,http,...,7,556,9,7092,-,6,5,8200.677959,0,5
115035,f115008,9.524471e+08,9.524471e+08,0.057650,172.16.115.5,24714,131.84.1.31,80,tcp,http,...,7,606,9,8032,-,6,5,8200.779714,0,5


## Misclassification Analysis

### Helper functions

In [49]:
def false_positives(df, phase):
    return df[(df["y_true"] != phase) & (df["y_pred"] == phase)]

def false_negatives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] != phase)]

def true_positives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] == phase)]

### Per-Phase Misclassifications

#### Phase 2

In [50]:
df_fp_2 = false_positives(mis_df, 2)
df_fp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
31662,f32132,9.524442e+08,9.524442e+08,0.0,172.16.112.194,0,197.218.177.69,8,icmp,-,...,1,38,0,0,-,1,0,5262.514702,2,0


In [51]:
df_fp_2_protocols = df_fp_2["proto"].value_counts()
print("False Positives for Phase 2 Protocol Distribution:")
print(df_fp_2_protocols)

False Positives for Phase 2 Protocol Distribution:
proto
icmp    1
Name: count, dtype: int64


In [52]:
df_fn_2 = false_negatives(mis_df, 2)
df_fn_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
24136,f24195,9.524417e+08,9.524417e+08,0.121346,202.77.162.213,54792,172.16.115.20,32773,udp,-,...,1,100,1,52,-,17,2,2790.882799,0,2
25735,f25926,9.524421e+08,9.524421e+08,0.057435,202.77.162.213,56256,172.16.112.10,32774,udp,-,...,1,100,1,52,-,17,2,3213.549749,0,2


In [53]:
df_fn_2["local_orig"]
df_fn_2["local_resp"]

24136    T
25735    T
Name: local_resp, dtype: object

In [54]:
df_tp_2 = true_positives(mis_df, 2)
df_tp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


#### Phase 3

In [55]:
df_fn_3 = false_negatives(mis_df, 3)
df_fn_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
29151,f29122,9.524433e+08,9.524433e+08,4.323372,202.77.162.213,46987,172.16.112.10,23,tcp,-,...,14,622,13,762,-,6,3,4392.243433,0,3
29157,f29150,9.524433e+08,9.524433e+08,8.326679,202.77.162.213,46988,172.16.112.50,23,tcp,-,...,14,622,12,627,-,6,3,4396.563364,0,3


In [56]:
df_fp_3 = false_positives(mis_df, 3)
df_fp_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


#### Phase 4

In [57]:
df_fn_4 = false_negatives(mis_df, 4)
df_fn_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [58]:
df_fp_4 = false_positives(mis_df, 4)
df_fp_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [59]:
df_fp_4_src_ips = df_fp_4["src_ip"].value_counts()
print("False Positives for Phase 4 Source IP Distribution:")
print(df_fp_4_src_ips)

False Positives for Phase 4 Source IP Distribution:
Series([], Name: count, dtype: int64)


In [60]:
df_fp_4["dport"].value_counts()

Series([], Name: count, dtype: int64)

#### Phase 5

In [61]:
# df_fn_5 = false_negatives(mis_df, 5)
# df_fn_5